In [1]:
# Cell: Optimized assignment (multi-source Dijkstra) implemented inline
from __future__ import annotations
import heapq
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling

# Minimal config (adjust paths if needed)
DATA_DIR = "dataset_big"
OUTPUT_PREFIX = "XX"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
WALK_FRICTION_RASTER = f"{DATA_DIR}/friction_walk.tif"
CAR_SHARE_RASTER = f"{DATA_DIR}/vehicles_allocation_share.tif"
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"

def _read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile

In [2]:
# Cell: Produce and save travel-time map using onstove RasterLayer.travel_time
import sys
from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio
from onstove.layer import RasterLayer

WALK_FRICTION_RASTER = f"{DATA_DIR}/friction_walk.tif"
resellers = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
with rasterio.open(WALK_FRICTION_RASTER) as src:
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width
rows, cols = rasterio.transform.rowcol(transform, resellers.geometry.x.values, resellers.geometry.y.values)
rows = np.asarray(rows, dtype=np.int32)
cols = np.asarray(cols, dtype=np.int32)
rows = rows[(rows >= 0) & (rows < height)]
cols = cols[(cols >= 0) & (cols < width)]
friction_layer = RasterLayer(category=None, name='walk_friction', path=WALK_FRICTION_RASTER)
traveltime_out = f"{OUTPUT_PREFIX}traveltime_map.tif"
_ = friction_layer.travel_time(rows=rows, cols=cols, include_starting_cells=False, output_path=f"{DATA_DIR}/{traveltime_out}", create_raster=True)
print("Saved travel time raster:", f"{DATA_DIR}/{traveltime_out}")

Saved travel time raster: dataset_big/XXtraveltime_map.tif


In [3]:
# Cell: Compare XX traveltime output with original Huff output
import os
import numpy as np
import pandas as pd
import rasterio

DATA_DIR = "dataset_big"
new_path = os.path.join(DATA_DIR, f"{OUTPUT_PREFIX}traveltime_preferred_distributor_per_pixel.tif")
old_path = os.path.join(DATA_DIR, "huff_preferred_distributor_per_pixel.tif")
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"

print('Verifying files...')
print('new:', new_path)
print('old:', old_path)

# (Rest of comparison is in the report cell)

Verifying files...
new: dataset_big\XXtraveltime_preferred_distributor_per_pixel.tif
old: dataset_big\huff_preferred_distributor_per_pixel.tif


In [4]:
# Cell: Report chiaro e grafici per il confronto tra XX (traveltime) e Huff
import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

DATA_DIR = "dataset_big"
new_path = os.path.join(DATA_DIR, f"{OUTPUT_PREFIX}traveltime_preferred_distributor_per_pixel.tif")
old_path = os.path.join(DATA_DIR, "huff_preferred_distributor_per_pixel.tif")
POPULATION_RASTER = os.path.join(DATA_DIR, "Population.tif")

print('Run this cell to produce the detailed report and CSVs.')

Run this cell to produce the detailed report and CSVs.
